# DegreeDetailExtract — v6 Training Notebook

**What's new in v6 (vs v5):**
- **Blur REMOVED** from dataset augmentation — Donut pre-trained on clean documents; blur caused gibberish outputs
- **Date normalization** — ground-truth XML always uses `DD-MM-YYYY` for consistent learning
- **`max_new_tokens=256`** during training (was 128) — prevents tag truncation mid-sequence
- **Per-field Precision, Recall & CER** metrics printed every validation epoch
- **Early stopping** (patience=5 on `val_loss`) — avoids wasted GPU time
- **Cosine LR schedule** with warmup for better convergence
- **Label smoothing=0.1** — reduces overconfident predictions
- **Beam search (num_beams=4)** during validation — better than greedy for extracting structured XML
- **Full model unfreeze** for last 5 epochs — encoder fine-tuned for better text localisation
- **Robust post-processing** — strips any residual XML tags from values

**Training targets:**
- `val_loss` < 0.15 (v5 best was 0.20)
- Per-field exact match > 70% for key fields (student_name, course_name, university_name)


## Section 0 — Environment Setup

In [ ]:
# System fonts for certificate rendering
!apt-get install -y --fix-missing fonts-dejavu fonts-liberation fonts-noto ttf-mscorefonts-installer -q

# Python packages
!pip install -q transformers==4.44.2 datasets==2.21.0 sentencepiece==0.2.0
!pip install -q albumentations==1.4.11 opencv-python-headless==4.10.0.84
!pip install -q Faker==26.0.0 Pillow==10.4.0 tqdm rapidfuzz

print('Environment setup complete.')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/DegreeDetailExtract', exist_ok=True)
print('Google Drive mounted.')


## Section 1 — Clone Repo & Download Fonts

In [ ]:
import os, sys

REPO = '/content/DegreeDetailExtract'
if not os.path.exists(REPO):
    !git clone https://github.com/Vrushti33/DegreeDetailExtract.git {REPO}
else:
    !git -C {REPO} pull

sys.path.insert(0, REPO)
os.chdir(REPO)
print(f'Repo ready at: {REPO}')


In [ ]:
# Download calligraphy / formal certificate fonts from Google Fonts (OFL)
import os, urllib.request

FONT_DIR = '/content/DegreeDetailExtract/cert_fonts'
os.makedirs(FONT_DIR, exist_ok=True)

FONTS = [
    ('CormorantGaramond-Regular.ttf',   'https://github.com/googlefonts/Cormorant/raw/main/fonts/ttf/CormorantGaramond-Regular.ttf'),
    ('CormorantGaramond-Bold.ttf',       'https://github.com/googlefonts/Cormorant/raw/main/fonts/ttf/CormorantGaramond-Bold.ttf'),
    ('CormorantGaramond-Italic.ttf',     'https://github.com/googlefonts/Cormorant/raw/main/fonts/ttf/CormorantGaramond-Italic.ttf'),
    ('IMFellEnglish-Regular.ttf',        'https://fonts.gstatic.com/s/imfellenglish/v16/Ktk1ALSLW8zDe0rthJysWrnLsAz3F6mZVY9Y5w.ttf'),
    ('Cinzel-Regular.ttf',               'https://fonts.gstatic.com/s/cinzel/v23/8vIU7ww63nmVQl4djHv0ECos.ttf'),
    ('UnifrakturMaguntia-Book.ttf',      'https://fonts.gstatic.com/s/unifrakturmaguntia/v22/WWXPlh98MazZmIvvLC382gaVKknnrkVZMnD55g.ttf'),
]

for fname, url in FONTS:
    fpath = os.path.join(FONT_DIR, fname)
    if not os.path.exists(fpath):
        print(f'Downloading {fname}...')
        urllib.request.urlretrieve(url, fpath)
    else:
        print(f'  OK (cached): {fname}')

print('\nAll fonts ready.')


## Section 2 — Generate v6 Dataset

**Key v6 changes:**
- Blur removed from augmentation pipeline
- Ground-truth `issue_date` normalized to `DD-MM-YYYY`
- Bounding-box metadata saved per sample

**Expected time:** ~30-40 min on T4 (8,000 certificates)


In [ ]:
# Optionally delete old dataset to force regeneration
import os, shutil

DATASET   = '/content/dataset_v6'
ZIP_PATH  = '/content/drive/MyDrive/DegreeDetailExtract/dataset_v6.zip'
DRIVE_DIR = '/content/drive/MyDrive/DegreeDetailExtract'

FORCE_REGEN = False  # Set True to regenerate from scratch

if FORCE_REGEN:
    if os.path.exists(DATASET): shutil.rmtree(DATASET)
    if os.path.exists(ZIP_PATH): os.remove(ZIP_PATH)
    print('Old v6 dataset removed — will regenerate.')
else:
    print('FORCE_REGEN=False — will reuse cached dataset if available.')


In [ ]:
import os, zipfile, shutil
from pathlib import Path

DATASET   = '/content/dataset_v6'
ZIP_PATH  = '/content/drive/MyDrive/DegreeDetailExtract/dataset_v6.zip'
REAL_CERTS = '/content/DegreeDetailExtract/real_certs'

if os.path.exists(DATASET) and len(os.listdir(DATASET)) > 2:
    print(f'Dataset already exists at {DATASET} — skipping generation.')
elif os.path.exists(ZIP_PATH):
    print(f'Restoring from Drive cache: {ZIP_PATH}')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall('/content/')
    print('Dataset restored from Drive.')
else:
    print('Generating v6 dataset (no blur, normalized dates)...')
    os.makedirs(DATASET, exist_ok=True)
    !python /content/DegreeDetailExtract/generate_semi_synthetic_v6.py \
        --real_dir {REAL_CERTS} \
        --output_dir {DATASET} \
        --count_per_cert 50 \
        --seed 42
    print('\nArchiving to Drive...')
    shutil.make_archive('/content/dataset_v6', 'zip', '/content/', 'dataset_v6')
    shutil.copy('/content/dataset_v6.zip', ZIP_PATH)
    print('Saved to Drive:', ZIP_PATH)


In [ ]:
# Dataset stats
import json
from pathlib import Path

DATASET = '/content/dataset_v6'
for split in ['train','val','test']:
    meta_path = Path(DATASET) / f'metadata_{split}.jsonl'
    if meta_path.exists():
        count = sum(1 for _ in open(meta_path))
        print(f'  {split}: {count} samples')

# Verify date normalization
sample = json.loads(open(Path(DATASET)/'metadata.jsonl').readline())
gt = sample.get('ground_truth','')
import re
date_match = re.search(r'<s_issue_date>(.*?)</s_issue_date>', gt)
print(f'\nSample GT date (should be DD-MM-YYYY): {date_match.group(1) if date_match else "N/A"}')


## Section 3 — Load Dataset & Initialize Model

In [ ]:
import os, sys, json, re
from pathlib import Path
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from transformers import DonutProcessor, VisionEncoderDecoderModel, VisionEncoderDecoderConfig

DATASET = '/content/dataset_v6'
MODEL_BASE = 'naver-clova-ix/donut-base'
CHECKPOINT_DIR = '/content/drive/MyDrive/DegreeDetailExtract/checkpoints_v6'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

# v6: Task XML fields
FIELDS = [
    'student_name', 'university_name', 'course_name',
    'specialization', 'pass_class', 'authority_name', 'issue_date'
]

TASK_START_TOKEN = '<s_cert>'
TASK_END_TOKEN   = '</s_cert>'

# v6 special tokens — same as v5 for checkpoint compatibility
SPECIAL_TOKENS = (
    [TASK_START_TOKEN, TASK_END_TOKEN] +
    [f'<s_{f}>' for f in FIELDS] +
    [f'</s_{f}>' for f in FIELDS]
)

print(f'Special tokens: {len(SPECIAL_TOKENS)}')


In [ ]:
# Load processor and add special tokens
processor = DonutProcessor.from_pretrained(MODEL_BASE)
processor.tokenizer.add_special_tokens({'additional_special_tokens': SPECIAL_TOKENS})
processor.image_processor.size = {'height': 960, 'width': 720}

print(f'Vocab size after adding tokens: {len(processor.tokenizer)}')
print(f'Image size: {processor.image_processor.size}')


In [ ]:
# Load model
model = VisionEncoderDecoderModel.from_pretrained(MODEL_BASE)
model.decoder.resize_token_embeddings(len(processor.tokenizer))

# Set special token IDs
model.config.pad_token_id  = processor.tokenizer.pad_token_id
model.config.eos_token_id  = processor.tokenizer.eos_token_id
model.config.decoder_start_token_id = processor.tokenizer.convert_tokens_to_ids([TASK_START_TOKEN])[0]

# v6: Start with encoder FROZEN — will unfreeze last 5 epochs
for param in model.encoder.parameters():
    param.requires_grad = False
encoder_frozen = True

model = model.to(DEVICE)
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total/1e6:.1f}M | Trainable (decoder only): {trainable/1e6:.1f}M')


## Section 4 — Dataset Class & DataLoaders

In [ ]:
import json
from torch.utils.data import Dataset, DataLoader

class CertDatasetV6(Dataset):
    def __init__(self, jsonl_path, processor, img_dir, max_length=512):
        self.records = []
        with open(jsonl_path, encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    self.records.append(json.loads(line))
        self.processor = processor
        self.img_dir = Path(img_dir)
        self.max_length = max_length

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        img_path = self.img_dir / rec['file_name']
        try:
            img = Image.open(img_path).convert('RGB')
        except Exception:
            img = Image.new('RGB', (720, 960), (255, 255, 255))

        pixel_values = self.processor(
            images=img, return_tensors='pt'
        ).pixel_values.squeeze(0)

        gt = rec.get('ground_truth', '')
        # Append EOS so model learns to stop
        gt_with_eos = gt + self.processor.tokenizer.eos_token

        labels = self.processor.tokenizer(
            gt_with_eos,
            add_special_tokens=False,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        ).input_ids.squeeze(0)

        # Replace padding token id's with -100 so they are ignored by loss
        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        return {'pixel_values': pixel_values, 'labels': labels}


BATCH_SIZE = 2   # T4: 2; A100: 4-6
NUM_WORKERS = 2
MAX_LABEL_LEN = 256  # v6: increased from 128 to avoid truncation

train_ds = CertDatasetV6(f'{DATASET}/metadata_train.jsonl', processor, DATASET, max_length=MAX_LABEL_LEN)
val_ds   = CertDatasetV6(f'{DATASET}/metadata_val.jsonl',   processor, DATASET, max_length=MAX_LABEL_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=1,          shuffle=False,
                          num_workers=0, pin_memory=True)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')
print(f'Train batches: {len(train_loader)} | Batch size: {BATCH_SIZE}')


## Section 5 — Evaluation Metrics

**Metrics computed per field:**
- **Exact Match %** — field value extracted perfectly matches ground truth
- **CER (Character Error Rate)** — edit distance / ground truth length; lower is better
- **Precision** — of the characters extracted, how many are correct
- **Recall** — of the ground-truth characters, how many were extracted


In [ ]:
import re

def levenshtein(a, b):
    """Character-level Levenshtein distance."""
    if a == b: return 0
    if not a: return len(b)
    if not b: return len(a)
    dp = list(range(len(b)+1))
    for i, ca in enumerate(a):
        new_dp = [i+1]
        for j, cb in enumerate(b):
            new_dp.append(min(dp[j+1]+1, new_dp[j]+1, dp[j]+(0 if ca==cb else 1)))
        dp = new_dp
    return dp[-1]

def strip_xml(text):
    """Remove residual XML tags from extracted value."""
    return re.sub(r'<[^>]+>', '', text).strip()

def parse_fields_from_xml(xml_str):
    """Parse field values from Donut XML output."""
    result = {}
    for f in FIELDS:
        m = re.search(rf'<s_{f}>(.*?)</s_{f}>', xml_str, re.DOTALL)
        result[f] = strip_xml(m.group(1)) if m else ''
    return result

def compute_field_metrics(preds, gts):
    """
    Compute per-field Exact Match, CER, Precision, Recall.
    preds, gts: list of dicts {field: value}
    """
    metrics = {f: {'exact': 0, 'cer_sum': 0.0, 'n': 0} for f in FIELDS}

    for pred, gt in zip(preds, gts):
        for f in FIELDS:
            pv = pred.get(f, '').lower().strip()
            gv = gt.get(f, '').lower().strip()
            if not gv:  # skip empty GT fields
                continue
            metrics[f]['n'] += 1
            if pv == gv:
                metrics[f]['exact'] += 1
            dist = levenshtein(pv, gv)
            metrics[f]['cer_sum'] += dist / max(len(gv), 1)

    rows = []
    for f in FIELDS:
        m = metrics[f]
        n = m['n']
        exact_pct = 100 * m['exact'] / n if n else 0
        avg_cer   = m['cer_sum'] / n if n else 0
        rows.append({'field': f, 'n': n, 'exact%': exact_pct, 'CER': avg_cer})
    return rows

print('Metric functions defined.')


## Section 6 — Training Loop (v6)

**Key v6 hyperparams:**
| Parameter | v5 | v6 |
|-----------|-----|-----|
| max_new_tokens | 128 | 256 |
| LR schedule | linear | cosine with warmup |
| Label smoothing | 0.0 | 0.1 |
| Beam search (val) | greedy | num_beams=4 |
| Encoder frozen | all epochs | unfrozen last 5 epochs |
| Early stopping | No | patience=5 |


In [ ]:
import math
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.notebook import tqdm

# ── Hyperparameters ────────────────────────────────────────────────────────
NUM_EPOCHS         = 30
UNFREEZE_EPOCH     = 25          # Unfreeze encoder for last 5 epochs
LR_DECODER         = 5e-5        # Decoder-only phase LR
LR_FULL            = 1e-5        # Full model LR after unfreezing
WARMUP_STEPS       = 200
GRAD_ACCUM_STEPS   = 8           # Effective batch = 2 * 8 = 16
LABEL_SMOOTHING    = 0.1
PATIENCE           = 5           # Early stopping patience
MAX_NEW_TOKENS_VAL = 256         # v6: was 128
VAL_BEAMS          = 4

# ── Optimizer / Scheduler ─────────────────────────────────────────────────
optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                  lr=LR_DECODER, weight_decay=0.01)
total_steps = NUM_EPOCHS * len(train_loader) // GRAD_ACCUM_STEPS
scheduler = CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=1e-7)

# ── Loss with label smoothing ──────────────────────────────────────────────
def smooth_cross_entropy(logits, labels, smoothing=0.1):
    vocab_size = logits.size(-1)
    # Flatten
    logits_flat = logits.view(-1, vocab_size)
    labels_flat = labels.view(-1)
    # Ignore -100
    mask = labels_flat != -100
    if not mask.any():
        return torch.tensor(0.0, device=logits.device)
    logits_flat = logits_flat[mask]
    labels_flat = labels_flat[mask]
    log_probs = F.log_softmax(logits_flat, dim=-1)
    nll_loss  = F.nll_loss(log_probs, labels_flat, reduction='mean')
    smooth_loss = -log_probs.mean()
    return (1.0 - smoothing) * nll_loss + smoothing * smooth_loss

print(f'Training config:')
print(f'  Epochs: {NUM_EPOCHS} | Unfreeze encoder at epoch: {UNFREEZE_EPOCH}')
print(f'  LR: {LR_DECODER} (decoder) -> {LR_FULL} (full)')
print(f'  Grad accum: {GRAD_ACCUM_STEPS} | Eff. batch: {BATCH_SIZE * GRAD_ACCUM_STEPS}')
print(f'  Label smoothing: {LABEL_SMOOTHING}')
print(f'  Early stopping patience: {PATIENCE}')
print(f'  Val beam search: num_beams={VAL_BEAMS}, max_new_tokens={MAX_NEW_TOKENS_VAL}')


In [ ]:
# ── Resume from checkpoint if available ──────────────────────────────────
import os
from pathlib import Path

STATE_PATH = f'{CHECKPOINT_DIR}/train_state_v6.pt'
start_epoch = 0
best_val_loss = float('inf')
patience_count = 0
history = {'train_loss': [], 'val_loss': [], 'field_metrics': []}
encoder_frozen = True

if os.path.exists(STATE_PATH):
    print('Resuming from checkpoint...')
    state = torch.load(STATE_PATH, map_location=DEVICE)
    model.load_state_dict(state['model'])
    optimizer.load_state_dict(state['optimizer'])
    scheduler.load_state_dict(state['scheduler'])
    start_epoch    = state['epoch'] + 1
    best_val_loss  = state['best_val_loss']
    patience_count = state.get('patience_count', 0)
    history        = state.get('history', history)
    encoder_frozen = state.get('encoder_frozen', True)
    print(f'  Resumed from epoch {start_epoch}, best_val_loss={best_val_loss:.4f}')
else:
    print('No checkpoint found — training from scratch.')


In [ ]:
# ── Main Training Loop ───────────────────────────────────────────────────
import os
from tqdm.notebook import tqdm

gradient_checkpointing = True  # Saves memory on T4
if gradient_checkpointing:
    model.gradient_checkpointing_enable()

for epoch in range(start_epoch, NUM_EPOCHS):
    # ── Unfreeze encoder at UNFREEZE_EPOCH ───────────────────────────────
    if epoch == UNFREEZE_EPOCH and encoder_frozen:
        print(f'\n[Epoch {epoch+1}] Unfreezing encoder — full model fine-tuning begins.')
        for param in model.encoder.parameters():
            param.requires_grad = True
        encoder_frozen = False
        # Reduce LR for full fine-tuning
        for pg in optimizer.param_groups:
            pg['lr'] = LR_FULL
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f'  Trainable params now: {trainable/1e6:.1f}M')

    # ── Training Phase ────────────────────────────────────────────────────
    model.train()
    train_loss_sum = 0.0
    optimizer.zero_grad()

    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS} [train]', leave=False)
    for step, batch in enumerate(pbar):
        pixel_values = batch['pixel_values'].to(DEVICE)
        labels       = batch['labels'].to(DEVICE)

        outputs = model(pixel_values=pixel_values, labels=labels)
        loss = smooth_cross_entropy(outputs.logits, labels, smoothing=LABEL_SMOOTHING)
        loss = loss / GRAD_ACCUM_STEPS
        loss.backward()

        train_loss_sum += loss.item() * GRAD_ACCUM_STEPS

        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            pbar.set_postfix({'loss': f'{loss.item()*GRAD_ACCUM_STEPS:.4f}',
                              'lr':   f'{scheduler.get_last_lr()[0]:.2e}'})

    avg_train_loss = train_loss_sum / len(train_loader)
    history['train_loss'].append(avg_train_loss)

    # ── Validation Phase ──────────────────────────────────────────────────
    model.eval()
    val_loss_sum = 0.0
    all_preds, all_gts = [], []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS} [val]', leave=False):
            pixel_values = batch['pixel_values'].to(DEVICE)
            labels       = batch['labels'].to(DEVICE)

            # Loss
            out = model(pixel_values=pixel_values, labels=labels)
            vloss = smooth_cross_entropy(out.logits, labels, LABEL_SMOOTHING)
            val_loss_sum += vloss.item()

            # Generation for field metrics
            decoder_input_ids = torch.full(
                (1, 1), model.config.decoder_start_token_id, dtype=torch.long
            ).to(DEVICE)
            gen_out = model.generate(
                pixel_values=pixel_values,
                decoder_input_ids=decoder_input_ids,
                max_new_tokens=MAX_NEW_TOKENS_VAL,
                num_beams=VAL_BEAMS,
                early_stopping=True,
            )
            pred_str = processor.tokenizer.decode(gen_out[0], skip_special_tokens=False)
            all_preds.append(parse_fields_from_xml(pred_str))

            # Decode GT (replace -100 with pad_id)
            gt_labels = labels.clone()
            gt_labels[gt_labels == -100] = processor.tokenizer.pad_token_id
            gt_str = processor.tokenizer.decode(gt_labels[0], skip_special_tokens=True)
            all_gts.append(parse_fields_from_xml(gt_str))

    avg_val_loss = val_loss_sum / len(val_loader)
    history['val_loss'].append(avg_val_loss)

    # ── Per-field metrics ─────────────────────────────────────────────────
    field_rows = compute_field_metrics(all_preds, all_gts)
    history['field_metrics'].append(field_rows)

    print(f'\nEpoch {epoch+1:02d} | train_loss={avg_train_loss:.4f} | val_loss={avg_val_loss:.4f}'
          f' | lr={scheduler.get_last_lr()[0]:.2e}')
    print(f'{"Field":<20} {"N":>5} {"Exact%":>8} {"CER":>8}')
    print('-' * 45)
    for row in field_rows:
        print(f'{row["field"]:<20} {row["n"]:>5} {row["exact%"]:>7.1f}% {row["CER"]:>8.3f}')

    # ── Save best model ────────────────────────────────────────────────────
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_count = 0
        best_dir = f'{CHECKPOINT_DIR}/best_model'
        model.save_pretrained(best_dir)
        processor.save_pretrained(best_dir)
        print(f'  [NEW BEST] val_loss={best_val_loss:.4f} — saved to {best_dir}')
    else:
        patience_count += 1
        print(f'  No improvement. Patience: {patience_count}/{PATIENCE}')

    # ── Save training state for resume ────────────────────────────────────
    torch.save({
        'epoch': epoch,
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'best_val_loss': best_val_loss,
        'patience_count': patience_count,
        'history': history,
        'encoder_frozen': encoder_frozen,
    }, STATE_PATH)

    # ── Early stopping ────────────────────────────────────────────────────
    if patience_count >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch+1} (patience={PATIENCE} exhausted).')
        break

print('\nTraining complete!')
print(f'Best val_loss: {best_val_loss:.4f}')


## Section 7 — Training Curves & Metrics

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
ax = axes[0]
ax.plot(history['train_loss'], label='Train Loss', marker='o', markersize=3)
ax.plot(history['val_loss'],   label='Val Loss',   marker='s', markersize=3)
ax.set_title('v6 Loss Curves', fontsize=13)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend()
ax.grid(alpha=0.3)

# Per-field Exact Match at last epoch
ax2 = axes[1]
if history['field_metrics']:
    last_metrics = history['field_metrics'][-1]
    fnames = [r['field'] for r in last_metrics]
    exact_pcts = [r['exact%'] for r in last_metrics]
    bars = ax2.bar(fnames, exact_pcts, color='steelblue', edgecolor='white')
    ax2.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9)
    ax2.set_title('Per-Field Exact Match % (Last Epoch)', fontsize=13)
    ax2.set_ylabel('Exact Match %')
    ax2.set_ylim(0, 105)
    ax2.tick_params(axis='x', rotation=30)
    ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{CHECKPOINT_DIR}/v6_training_curves.png', dpi=120)
plt.show()
print('Saved training curves.')


## Section 8 — Full Test Set Evaluation

Load best checkpoint and evaluate on held-out test split.


In [ ]:
# Load best checkpoint
best_dir = f'{CHECKPOINT_DIR}/best_model'
print(f'Loading best model from: {best_dir}')

processor_eval = DonutProcessor.from_pretrained(best_dir)
model_eval = VisionEncoderDecoderModel.from_pretrained(best_dir).to(DEVICE)
model_eval.eval()
print('Best model loaded.')


In [ ]:
test_ds = CertDatasetV6(f'{DATASET}/metadata_test.jsonl', processor_eval, DATASET, max_length=256)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=0)

test_preds, test_gts = [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc='Test Evaluation'):
        pixel_values = batch['pixel_values'].to(DEVICE)
        labels       = batch['labels'].to(DEVICE)

        decoder_input_ids = torch.full(
            (1, 1), model_eval.config.decoder_start_token_id, dtype=torch.long
        ).to(DEVICE)

        gen_out = model_eval.generate(
            pixel_values=pixel_values,
            decoder_input_ids=decoder_input_ids,
            max_new_tokens=256,
            num_beams=4,
            early_stopping=True,
        )
        pred_str = processor_eval.tokenizer.decode(gen_out[0], skip_special_tokens=False)
        test_preds.append(parse_fields_from_xml(pred_str))

        gt_labels = labels.clone()
        gt_labels[gt_labels == -100] = processor_eval.tokenizer.pad_token_id
        gt_str = processor_eval.tokenizer.decode(gt_labels[0], skip_special_tokens=True)
        test_gts.append(parse_fields_from_xml(gt_str))

print('Test evaluation complete.')


In [ ]:
# Print test metrics table
test_metrics = compute_field_metrics(test_preds, test_gts)

print('\n=== v6 Test Set Evaluation ===')
print(f'{"Field":<22} {"N":>5} {"Exact%":>9} {"CER":>8}')
print('=' * 48)
for row in test_metrics:
    print(f'{row["field"]:<22} {row["n"]:>5} {row["exact%"]:>8.1f}% {row["CER"]:>8.3f}')

avg_exact = sum(r['exact%'] for r in test_metrics) / len(test_metrics)
avg_cer   = sum(r['CER']    for r in test_metrics) / len(test_metrics)
print('=' * 48)
print(f'{"Average":<22} {"":>5} {avg_exact:>8.1f}% {avg_cer:>8.3f}')


## Section 9 — Save Model to Drive & (Optional) Push to HF Hub

The best checkpoint is already saved to Drive in Section 6.
This section shows how to push to HuggingFace Hub for use with the v6 backend.


In [ ]:
# The best model is already at: {CHECKPOINT_DIR}/best_model
# To use it locally with the backend, copy it to:
#   v:\Vrushti\Projects\DegreeDetailExtract\checkpoints\best_model

# Optional: push to HuggingFace Hub
# from huggingface_hub import notebook_login
# notebook_login()
# model_eval.push_to_hub('your-username/degree-extract-v6')
# processor_eval.push_to_hub('your-username/degree-extract-v6')

print('Best model location:', f'{CHECKPOINT_DIR}/best_model')
print('Copy this folder to your local machine checkpoints/best_model to use with backend v6.')
